# Homework - Everything & Then Some

## Section 3 - Feature Selection

### 3.0 Environment Setup & Data Loading

To keep our workflow structured and reproducible, we begin by importing necessary libraries and defining our target variable. We set a fixed seed (SEED = 42) across all experiments to ensure consistent results.

We load the clean dataset from our preprocessing phase (train_clean.csv) and separate our target, repeat_purchase_90d.

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from scipy.stats import chi2_contingency
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import LogisticRegression

# Set random seed for full reproducibility
SEED = 42
np.random.seed(SEED)

TARGET = "repeat_purchase_90d"

# Load clean training dataset
train_clean = pd.read_csv("../hdata/train_clean.csv")
df_feat = train_clean.copy()

# Target vector
y = df_feat[TARGET].astype(int)

### 3.1 Critical Removal of Identifiers and High-Cardinality Leakage

Before applying any automated algorithms, we critically examine our variables for potential data leakage:

Primary Identifiers (ID, customer_id): Unique keys or row indexes serve no predictive purpose and cause models to memorize sequence positions or individual customer identities.

Synthetic Noise (random_noise): Included during early experimentation as an empirical baseline. Any true predictor must outperform random noise.

Date Features (purchase_date): Raw timestamps lead to temporal extrapolation errors when predicting future test periods.

We exclude these columns upfront to ensure our candidate feature matrix consists purely of generalizable attributes.

In [23]:
# Identify non-predictive variables to drop upfront
identifiers_and_noise = [
    "ID",
    "customer_id",
    "random_noise",
    "purchase_date",
    "purchase_date_dt",
]

# Extract candidate numerical features
candidate_cols = [
    c
    for c in df_feat.select_dtypes(include=["number"]).columns
    if c not in [TARGET] + identifiers_and_noise
]

# Impute missing values with column medians for ranking algorithms
X_num = df_feat[candidate_cols].fillna(df_feat[candidate_cols].median())

print(f"Candidate numerical features: {X_num.shape[1]}")

Candidate numerical features: 36


### 3.2 Statistical Filter Methods: Correlation & ANOVA F-Score

We evaluate our continuous numerical predictors using two statistical filter metrics: Point-Biserial Correlation: Measures the linear relationship between continuous attributes and our binary target. ANOVA F-Score: Measures the ratio of between-group variance to within-group variance ($y=0$ vs. $y=1$). A higher F-score indicates a stronger signal-to-noise ratio. Combining both metrics gives us a robust baseline view of individual feature importance.

In [24]:

# 1. Point-Biserial Correlation
# Measures the straight-line relationship between each numeric column and our target (0 or 1).
# Calculate the correlation for each column and take the absolute value (making negatives positive)
abs_correlations = X_num.corrwith(y).abs()

# Put the results into a clean, easy-to-read table
filter_corr_df = pd.DataFrame(
    {"feature": candidate_cols, "abs_corr": abs_correlations}
).reset_index(drop=True)


# 2. ANOVA F-Score
# Compares the average values between buyers (y=1) and non-buyers (y=0).
# A higher score means the feature separates the two groups more clearly.


def compute_anova_f(column, target):
    # Split the column into two groups based on the target
    group_0 = column[target == 0]
    group_1 = column[target == 1]

    # Calculate average (mean) values for each group and overall
    mean_0 = group_0.mean()
    mean_1 = group_1.mean()
    overall_mean = column.mean()

    # Variance between the two groups
    ss_between = len(group_0) * (mean_0 - overall_mean) ** 2 + len(
        group_1
    ) * (mean_1 - overall_mean) ** 2
    ms_between = ss_between / 1.0  # (number of groups - 1) = 2 - 1 = 1

    # Variance within each group
    ss_within = ((group_0 - mean_0) ** 2).sum() + (
        (group_1 - mean_1) ** 2
    ).sum()
    ms_within = ss_within / (len(column) - 2)

    # Return the F-score ratio (Signal / Noise)
    if ms_within > 0:
        return ms_between / ms_within
    else:
        return 0.0


# Calculate the ANOVA F-score for every column
f_scores = X_num.apply(lambda col: compute_anova_f(col, y))

# Put the F-scores into a clean table
filter_f_df = pd.DataFrame(
    {"feature": candidate_cols, "f_score": f_scores}
).reset_index(drop=True)

### 3.3 Collinearity & Redundancy Pruning (|r| > 0.85)

Many engineered features carry duplicate information (for example, 90-day spend vs. 180-day spend). High collinearity inflates coefficient variance in linear models and adds unnecessary complexity. We scan all pairwise correlations. If two features exhibit a correlation higher than $\vert{}r\vert{} = 0.85$, we keep the feature with the superior ANOVA F-score and drop the redundant variable.

In [25]:
corr_matrix = X_num.corr().abs()
threshold = 0.85
to_drop_collinear = set()

for i in range(len(candidate_cols)):
    for j in range(i + 1, len(candidate_cols)):
        col1, col2 = candidate_cols[i], candidate_cols[j]
        if corr_matrix.loc[col1, col2] > threshold:
            # Retain feature with higher ANOVA F-score
            if f_scores[col1] >= f_scores[col2]:
                to_drop_collinear.add(col2)
            else:
                to_drop_collinear.add(col1)

print(
    f"Redundant features dropped (|r| > {threshold}): {list(to_drop_collinear)}"
)

Redundant features dropped (|r| > 0.85): ['recent_activity_ratio', 'Total a Pagar', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'Total Liquido']


### 3.4 Categorical Feature Filtering (Chi-Square Test)

To select meaningful categorical features without introducing high-cardinality noise (such as individual transport routes with 70+ categories), we run a Chi-Square Test ($\chi^2$). We evaluate low-cardinality categoricals (under 50 distinct levels) against our target. Only categories with a statistically significant association ($p < 0.05$) are retained.

In [26]:
cat_cols = [
    c
    for c in df_feat.select_dtypes(include=["object", "category"]).columns
    if c not in identifiers_and_noise
]
selected_categoricals = []

for col in cat_cols:
    if df_feat[col].nunique() < 50:  # Avoid high-cardinality leakage
        table = pd.crosstab(df_feat[col].fillna("Missing"), y)
        chi2, p_val, _, _ = chi2_contingency(table)
        if p_val < 0.05:  # Keep statistically significant categoricals
            selected_categoricals.append(col)

print(f"Statistically significant categoricals: {selected_categoricals}")

Statistically significant categoricals: ['payment_type', 'salesperson', 'commercial_zone', 'warehouse', 'purchase_dayofweek', 'country_or_region', 'commercial_region', 'island']


### 3.5 Consensus Ranking & Comparison with RFE and L1 Lasso

To ensure our selected feature subset is well-rounded, we calculate a Consensus Rank Score by averaging the feature ranks from both Correlation and ANOVA F-Score metrics. For comparison and validation, we also run: 

- Wrapper Method (RFE): Recursively eliminates least important variables using Logistic Regression.

- Embedded Method ($L_1$ Lasso): Shrinks less important coefficients directly to zero.

In [27]:

# Step 1: Remove Redundant (Collinear) Features
# Create a cleaned dataset of numbers by dropping the redundant columns we found earlier
X_filtered_num = X_num.drop(columns=list(to_drop_collinear))

# Step 2: Build a Consensus Ranking Matrix (Combining Filter Methods)
# Combine our Correlation and ANOVA F-Score results into a single table
selection_matrix = pd.merge(filter_corr_df, filter_f_df, on="feature")

# Remove the redundant features from this summary table too
keep_mask = ~selection_matrix["feature"].isin(to_drop_collinear)
selection_matrix = selection_matrix[keep_mask].copy()

# Rank features for each metric (1 = highest score, 2 = second highest, etc.)
selection_matrix["rank_corr"] = selection_matrix["abs_corr"].rank(
    ascending=False
)
selection_matrix["rank_f"] = selection_matrix["f_score"].rank(ascending=False)

# Average the two ranks to create a final consensus score (lower score = better feature)
selection_matrix["consensus_score"] = (
    selection_matrix["rank_corr"] + selection_matrix["rank_f"]
) / 2.0

# Sort the table so the best consensus features are at the top
selection_matrix = selection_matrix.sort_values(
    by="consensus_score"
).reset_index(drop=True)

# Pick the top 15 features from our consensus ranking
consensus_features = selection_matrix.head(15)["feature"].tolist()


# Step 3: Wrapper Method — Recursive Feature Elimination (RFE)
# Set up a Logistic Regression model to test features in combinations
base_model = LogisticRegression(max_iter=1000, random_state=SEED)

# RFE repeatedly fits the model and removes the 2 weakest features until only 15 remain
rfe_selector = RFE(
    estimator=base_model,
    n_features_to_select=15,
    step=2,
)
rfe_selector.fit(X_filtered_num, y)

# Get the names of the 15 features selected by RFE
rfe_features = X_filtered_num.columns[rfe_selector.get_support()].tolist()


# Step 4: Embedded Method — L1 Regularization (Lasso)


# L1 (Lasso) penalty automatically shrinks unhelpful feature coefficients down to zero
l1_logistic = LogisticRegression(
    penalty="l1",  # Apply L1 regularization
    solver="liblinear",  # Solver required for L1
    C=0.1,  # Strength of regularization (smaller C = stronger penalty)
    max_iter=1000,
    random_state=SEED,
)

# SelectFromModel keeps only features whose coefficients were NOT shrunk to zero
l1_selector = SelectFromModel(estimator=l1_logistic, threshold=1e-8)
l1_selector.fit(X_filtered_num, y)

# Get the names of the features selected by L1 Lasso
l1_features = X_filtered_num.columns[l1_selector.get_support()].tolist()

### 3.6 Final Selection & Justification

Our final feature set combines the Top 15 Consensus Numerical Attributes with the Statistically Significant Categorical Variables.

**Summary of Key Drivers Retained:**

- **Short-Term Recency & Frequency:** `purchase_count_365d`, `purchase_count_30d`, `spend_180d`, and `spend_30d` consistently rank at the top, proving recency is the strongest signal for 90-day repurchase behavior.

- **Customer Baseline Attributes:** `prior_purchase_count`, `prior_total_spend`, `days_since_previous_purchase`, and `customer_tenure_days` distinguish established repeat customers from first-time buyers.

- **Behavioral Metrics:** `discount_rate` and `purchase_rate_per_year` capture overall customer engagement habits.

In [28]:
print("\nTOP 15 CONSENSUS NUMERICAL FEATURES ")
print(
    selection_matrix[["feature", "consensus_score", "abs_corr", "f_score"]]
    .head(15)
    .to_string(index=False)
)

FINAL_SELECTED_FEATURES = consensus_features + selected_categoricals
print(
    f"\nFinal Selected Feature Space ({len(FINAL_SELECTED_FEATURES)} total features):"
)
print(FINAL_SELECTED_FEATURES)


TOP 15 CONSENSUS NUMERICAL FEATURES 
                     feature  consensus_score  abs_corr    f_score
         purchase_count_365d              1.0  0.340343 847.231887
                  spend_180d              2.0  0.301013 644.349775
        prior_purchase_count              3.0  0.298811 634.035975
          purchase_count_30d              4.0  0.263339 481.887008
           prior_total_spend              5.0  0.261934 476.383042
               discount_rate              6.0  0.221933 335.029275
      purchase_rate_per_year              7.0  0.210093 298.629128
         prior_mean_gap_days              8.0  0.195046 255.752741
                  spend_365d              9.0  0.183368 225.009793
days_since_previous_purchase             10.0  0.169630 191.596863
                   spend_30d             11.0  0.165181 181.400251
          prior_std_gap_days             12.0  0.153751 156.577929
           is_first_purchase             13.0  0.146232 141.310615
    prior_std_purchase_v

### 3.7 Exporting the Final Feature Set

We combine our top consensus numerical features with our statistically significant categorical variables into a single list.
Finally, we filter our dataset and save the selected features to a new file (train_selected.csv).

In [29]:

# Step 1: Combine Numerical and Categorical Features
FINAL_SELECTED_FEATURES = consensus_features + selected_categoricals

print(
    f"Total features selected for modeling: {len(FINAL_SELECTED_FEATURES)}"
)
print("\nFinal Feature List:")
print(FINAL_SELECTED_FEATURES)


# Step 2: Create and Save the Final Selected Dataset
# Filter the original dataframe to keep only our selected features and the target
final_df = df_feat[FINAL_SELECTED_FEATURES + [TARGET]]

# Save to a CSV file for the next section
final_df.to_csv("../hdata/train_selected.csv", index=False)

print("\nSuccess! Final selected feature dataset saved to 'train_selected.csv'.")

Total features selected for modeling: 23

Final Feature List:
['purchase_count_365d', 'spend_180d', 'prior_purchase_count', 'purchase_count_30d', 'prior_total_spend', 'discount_rate', 'purchase_rate_per_year', 'prior_mean_gap_days', 'spend_365d', 'days_since_previous_purchase', 'spend_30d', 'prior_std_gap_days', 'is_first_purchase', 'prior_std_purchase_value', 'customer_tenure_days', 'payment_type', 'salesperson', 'commercial_zone', 'warehouse', 'purchase_dayofweek', 'country_or_region', 'commercial_region', 'island']

Success! Final selected feature dataset saved to 'train_selected.csv'.


### 3.8 Conclusion & Summary of Feature Selection

In this section, we built a structured feature selection pipeline to reduce our dataset from **137 features** down to a compact, highly predictive subset. This helps prevent overfitting, reduces model complexity, and removes noise.

### Summary of Our Steps:
1. **Removed Leakage & Identifiers:** We dropped unique IDs (`ID`, `customer_id`), raw timestamps (`purchase_date`), and synthetic noise (`random_noise`) to prevent our models from memorizing specific row sequences.

2. **Eliminated Redundant Features:** We checked pairwise linear correlations ($|r| > 0.85$). When two features measured the exact same thing, we kept the stronger one (based on its ANOVA F-Score) and dropped the duplicate.

3. **Filtered Categorical Variables:** Using a Chi-Square test of independence ($\chi^2$), we kept only categorical variables that showed a statistically significant relationship ($p < 0.05$) with repeat purchasing behavior.

4. **Combined Selection Methods:** We calculated a **Consensus Score** averaging correlation and ANOVA ranks, and compared our top picks with **RFE** (Wrapper) and **L1 Lasso** (Embedded) models.


### Key Predictive Drivers Retained:
* **Recency & Recent Activity:** Short-term metrics like `purchase_count_30d`, `spend_30d`, `purchase_count_365d`, and `spend_180d` proved to be the strongest overall signals for predicting whether a customer will buy again in 90 days.

* **Customer Purchase History:** Long-term attributes like `prior_purchase_count`, `prior_total_spend`, and `customer_tenure_days` ensure our model can distinguish loyal repeat customers from brand-new shoppers.

* **Engagement Patterns:** Customer behavior metrics like `discount_rate` and `purchase_rate_per_year` capture overall buying habits.